# Food Delivery Time Prediction System Using Linear Regression

**Academic Data Science Project**

---

## 1. Project Overview & Problem Statement
In on-demand food delivery platforms (such as Zomato, Swiggy, UberEats, and DoorDash), accurate estimation of food delivery time is critical for customer satisfaction, kitchen dispatch timing, and delivery fleet optimization.

### Objective:
Build a machine learning predictive system using **Linear Regression** to predict the total delivery time (in minutes) based on key operational variables:
- **Delivery distance (km)**
- **Preparation time (minutes)**
- **Delivery partner experience (years)**
- **Traffic conditions** (Low, Medium, High, Jam)
- **Weather conditions** (Clear, Rainy, Foggy, Windy, Stormy)
- **Time of day** (Morning, Afternoon, Evening, Night)
- **Vehicle type** (Motorcycle, Scooter, Electric Bike, Bicycle)

### Technology Stack:
- **Python** (Pandas, NumPy, Matplotlib, Seaborn, Scikit-learn)
- **Linear Regression** (Simple & Multiple Linear Regression)

## 2. Importing Required Libraries
We import the standard Data Science stack: Pandas for tabular manipulation, NumPy for numerical operations, Matplotlib and Seaborn for statistical plotting, and Scikit-learn for model building and evaluation.

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Scikit-learn utilities
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# Visual formatting settings
sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (10, 6)
plt.rcParams["font.size"] = 11
print("Libraries successfully imported!")

## 3. Loading the Dataset & Initial Exploration
We load the local dataset from `../dataset/delivery_data.csv` and inspect its structure, dimensions, and initial summary.

In [ ]:
# Load dataset
data_path = os.path.join("..", "dataset", "delivery_data.csv")
df = pd.read_csv(data_path)
print(f"Dataset Shape: {df.shape[0]} rows, {df.shape[1]} columns")
df.head()

In [ ]:
# Dataset info and data types
df.info()

In [ ]:
# Descriptive statistics for numerical variables
df.describe().T

## 4. Data Preprocessing & Cleaning
Real-world datasets frequently contain missing values and duplicates. In this section, we:
1. Identify and remove duplicate rows.
2. Check for missing values (`NaN`).
3. Impute missing numerical data using the **median** (robust to outliers).
4. Impute missing categorical data using the **mode** (most frequent category).

In [ ]:
# Check for duplicates
duplicates_count = df.duplicated().sum()
print(f"Number of duplicate rows found: {duplicates_count}")

if duplicates_count > 0:
    df = df.drop_duplicates()
    print(f"Duplicates removed! New shape: {df.shape}")

In [ ]:
# Inspect missing values
print("Missing values per feature:")
print(df.isnull().sum())

In [ ]:
# Impute Partner_Experience_years with median
med_exp = df["Partner_Experience_years"].median()
df["Partner_Experience_years"] = df["Partner_Experience_years"].fillna(med_exp)

# Impute Weather with mode
mode_weather = df["Weather"].mode()[0]
df["Weather"] = df["Weather"].fillna(mode_weather)

print("Missing values after imputation:")
print(df.isnull().sum())

## 5. Exploratory Data Analysis (EDA)
We explore the underlying relationships between the independent variables and the target variable (`Delivery_Time_min`).

In [ ]:
# 5.1 Distribution of Delivery Times
plt.figure(figsize=(9, 5))
sns.histplot(df["Delivery_Time_min"], kde=True, color="#2b5c8f", bins=30)
plt.title("Distribution of Food Delivery Times (Minutes)", fontsize=14, weight="bold")
plt.xlabel("Delivery Time (min)")
plt.ylabel("Frequency")
mean_val = df["Delivery_Time_min"].mean()
med_val = df["Delivery_Time_min"].median()
plt.axvline(mean_val, color="red", linestyle="--", label=f"Mean: {mean_val:.1f} min")
plt.axvline(med_val, color="green", linestyle=":", label=f"Median: {med_val:.1f} min")
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
# 5.2 Delivery Distance vs Delivery Time
plt.figure(figsize=(9, 5))
sns.regplot(data=df, x="Delivery_Distance_km", y="Delivery_Time_min",
            scatter_kws={"alpha": 0.4, "color": "#1f77b4"},
            line_kws={"color": "#d62728", "linewidth": 2.5})
plt.title("Delivery Distance vs Delivery Time", fontsize=14, weight="bold")
plt.xlabel("Distance (km)")
plt.ylabel("Delivery Time (min)")
plt.tight_layout()
plt.show()

In [ ]:
# 5.3 Traffic Conditions vs Delivery Time
traffic_order = ["Low", "Medium", "High", "Jam"]
plt.figure(figsize=(9, 5))
sns.boxplot(data=df, x="Traffic_Level", y="Delivery_Time_min", order=traffic_order, palette="Blues_d")
plt.title("Impact of Traffic Level on Delivery Time", fontsize=14, weight="bold")
plt.xlabel("Traffic Level")
plt.ylabel("Delivery Time (min)")
plt.tight_layout()
plt.show()

In [ ]:
# 5.4 Weather Conditions vs Delivery Time
weather_order = ["Clear", "Windy", "Foggy", "Rainy", "Stormy"]
plt.figure(figsize=(9, 5))
sns.barplot(data=df, x="Weather", y="Delivery_Time_min", order=weather_order, palette="flare", errorbar=None)
plt.title("Average Delivery Time Across Weather Conditions", fontsize=14, weight="bold")
plt.xlabel("Weather Condition")
plt.ylabel("Average Delivery Time (min)")
plt.tight_layout()
plt.show()

In [ ]:
# 5.5 Order Preparation Time vs Delivery Time
plt.figure(figsize=(9, 5))
sns.scatterplot(data=df, x="Preparation_Time_min", y="Delivery_Time_min", hue="Vehicle_Type", alpha=0.6, palette="tab10")
plt.title("Preparation Time vs Delivery Time (by Vehicle Type)", fontsize=14, weight="bold")
plt.xlabel("Preparation Time (min)")
plt.ylabel("Delivery Time (min)")
plt.tight_layout()
plt.show()

In [ ]:
# 5.6 Correlation Heatmap
num_cols = ["Delivery_Distance_km", "Preparation_Time_min", "Partner_Experience_years", "Delivery_Time_min"]
corr = df[num_cols].corr()
plt.figure(figsize=(8, 6))
sns.heatmap(corr, annot=True, cmap="coolwarm", fmt=".3f", vmin=-1, vmax=1, linewidths=0.5)
plt.title("Correlation Heatmap of Numerical Features", fontsize=14, weight="bold")
plt.tight_layout()
plt.show()

## 6. Simple Linear Regression (Baseline Model)
To understand the mathematical principles of Linear Regression, we first build a **Simple Linear Regression** model using a single predictor:

$$y = mx + c$$

where:
- $y$ = Delivery Time (min)
- $x$ = Delivery Distance (km)
- $m$ = Slope (how much time increases per km)
- $c$ = Intercept (baseline time when distance is 0)

In [ ]:
# Train Simple Linear Regression
X_simple = df[["Delivery_Distance_km"]]
y = df["Delivery_Time_min"]

X_train_s, X_test_s, y_train_s, y_test_s = train_test_split(X_simple, y, test_size=0.2, random_state=42)

slr = LinearRegression()
slr.fit(X_train_s, y_train_s)
y_pred_s = slr.predict(X_test_s)

# Compute metrics
mae_s = mean_absolute_error(y_test_s, y_pred_s)
mse_s = mean_squared_error(y_test_s, y_pred_s)
rmse_s = np.sqrt(mse_s)
r2_s = r2_score(y_test_s, y_pred_s)

print(f"Simple Linear Regression Equation: Delivery_Time = {slr.coef_[0]:.2f} * Distance + {slr.intercept_:.2f}")
print(f"MAE:  {mae_s:.2f} min")
print(f"MSE:  {mse_s:.2f}")
print(f"RMSE: {rmse_s:.2f} min")
print(f"R2 Score: {r2_s:.4f} ({r2_s*100:.1f}% variance explained)")

In [ ]:
# Visualizing Simple Linear Regression Line
plt.figure(figsize=(9, 5))
plt.scatter(X_test_s, y_test_s, color="#3b528b", alpha=0.5, label="Actual Test Data")
plt.plot(X_test_s, y_pred_s, color="#d62728", linewidth=2.5, label=f"Fitted Line (y = {slr.coef_[0]:.2f}x + {slr.intercept_:.2f})")
plt.title("Simple Linear Regression: Distance vs Delivery Time", fontsize=14, weight="bold")
plt.xlabel("Delivery Distance (km)")
plt.ylabel("Delivery Time (min)")
plt.legend()
plt.tight_layout()
plt.show()

## 7. Multiple Linear Regression (Primary Model)
Delivery time in real life depends simultaneously on distance, prep time, courier experience, traffic, weather, time of day, and vehicle type.

$$y = \beta_0 + \beta_1 X_1 + \beta_2 X_2 + \dots + \beta_p X_p + \epsilon$$

Categorical variables are encoded into binary dummy columns using **One-Hot Encoding**.

In [ ]:
# Feature Matrix and Target
num_features = ["Delivery_Distance_km", "Preparation_Time_min", "Partner_Experience_years"]
cat_features = ["Traffic_Level", "Weather", "Time_of_Day", "Vehicle_Type"]

X = df[num_features + cat_features]
y = df["Delivery_Time_min"]

# 80-20 Train-Test Split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print(f"Training instances: {X_train.shape[0]}, Testing instances: {X_test.shape[0]}")

# Preprocessor Pipeline
preprocessor = ColumnTransformer(
    transformers=[
        ("num", "passthrough", num_features),
        ("cat", OneHotEncoder(drop="first", sparse_output=False), cat_features)
    ]
)

model_pipeline = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("regressor", LinearRegression())
])

# Fit model on training data
model_pipeline.fit(X_train, y_train)
y_pred_m = model_pipeline.predict(X_test)
y_train_pred_m = model_pipeline.predict(X_train)

# Calculate metrics
mae_m = mean_absolute_error(y_test, y_pred_m)
mse_m = mean_squared_error(y_test, y_pred_m)
rmse_m = np.sqrt(mse_m)
r2_m = r2_score(y_test, y_pred_m)
r2_train_m = r2_score(y_train, y_train_pred_m)

print("\n--- Multiple Linear Regression Results ---")
print(f"Mean Absolute Error (MAE):     {mae_m:.2f} minutes")
print(f"Mean Squared Error (MSE):      {mse_m:.2f}")
print(f"Root Mean Squared Error (RMSE): {rmse_m:.2f} minutes")
print(f"Test R-squared (R2):           {r2_m:.4f} ({r2_m*100:.1f}%)")
print(f"Train R-squared (R2):          {r2_train_m:.4f} ({r2_train_m*100:.1f}%)")

In [ ]:
# Comparison between Simple and Multiple Linear Regression
comparison_df = pd.DataFrame({
    "Metric": ["MAE (minutes)", "MSE", "RMSE (minutes)", "R2 Score"],
    "Simple Linear Regression": [f"{mae_s:.2f}", f"{mse_s:.2f}", f"{rmse_s:.2f}", f"{r2_s:.4f}"],
    "Multiple Linear Regression": [f"{mae_m:.2f}", f"{mse_m:.2f}", f"{rmse_m:.2f}", f"{r2_m:.4f}"]
})
comparison_df

In [ ]:
# 7.1 Actual vs. Predicted Delivery Time Plot
plt.figure(figsize=(8, 8))
plt.scatter(y_test, y_pred_m, color="#1b7837", alpha=0.6, edgecolors="w", s=70, label="Predicted vs Actual")
min_val = min(y_test.min(), y_pred_m.min())
max_val = max(y_test.max(), y_pred_m.max())
plt.plot([min_val, max_val], [min_val, max_val], color="red", linestyle="--", linewidth=2, label="Perfect Prediction (y = x)")
plt.title("Multiple Linear Regression: Actual vs Predicted Delivery Time", fontsize=14, weight="bold")
plt.xlabel("Actual Delivery Time (min)")
plt.ylabel("Predicted Delivery Time (min)")
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
# 7.2 Residuals Distribution Plot
residuals = y_test - y_pred_m
plt.figure(figsize=(9, 5))
sns.histplot(residuals, kde=True, color="#762a83", bins=25)
plt.axvline(0, color="black", linestyle="--", linewidth=1.5)
plt.title("Residuals (Errors) Distribution (Actual - Predicted)", fontsize=14, weight="bold")
plt.xlabel("Residual Error (minutes)")
plt.ylabel("Count")
plt.tight_layout()
plt.show()

In [ ]:
# 7.3 Feature Coefficients Impact Bar Chart
cat_encoder = model_pipeline.named_steps["preprocessor"].named_transformers_["cat"]
cat_features_encoded = list(cat_encoder.get_feature_names_out(cat_features))
all_features = num_features + cat_features_encoded
reg = model_pipeline.named_steps["regressor"]

coef_df = pd.DataFrame({
    "Feature": all_features,
    "Coefficient": reg.coef_
}).sort_values(by="Coefficient")

plt.figure(figsize=(10, 7))
colors = ["#d73027" if c > 0 else "#4575b4" for c in coef_df["Coefficient"]]
plt.barh(coef_df["Feature"], coef_df["Coefficient"], color=colors)
plt.axvline(0, color="black", linewidth=1)
plt.title("Impact of Each Feature on Delivery Time (Coefficients)", fontsize=14, weight="bold")
plt.xlabel("Coefficient (Change in Delivery Minutes)")
plt.ylabel("Feature")
plt.tight_layout()
plt.show()

## 8. Summary of Findings & Viva Questions

### Key Findings:
1. **Multiple Linear Regression vs Simple Linear Regression:** Distance alone only explains ~28.5% of variance, whereas adding traffic, weather, prep time, vehicle type, and experience increases $R^2$ to over **93%** and drops MAE to ~2.2 minutes.
2. **Traffic & Weather:** Stormy weather adds ~13.7 minutes; Traffic Jam adds ~7.5-10 minutes.
3. **Preparation Time:** Directly adds ~1.0 minute per minute of food preparation.
4. **Vehicle:** Faster vehicles like motorcycles save significant transit time compared to bicycles.
5. **Partner Experience:** Experienced couriers reduce delivery time by ~0.77 minutes per year of experience.

### Common Viva Questions & Answers:
- **Q1: Why use Linear Regression for this problem?**
  *Answer:* Food delivery time has natural additive components: $Delivery\ Time = Prep\ Time + Travel\ Time + Traffic\ Delay + Weather\ Delay$. Linear Regression provides direct interpretability of how each factor adds or subtracts minutes.

- **Q2: What is the difference between MAE and RMSE?**
  *Answer:* MAE is the average absolute error in minutes. RMSE squares errors before averaging, penalizing larger mistakes more heavily.

- **Q3: What does the R-squared ($R^2$) score represent?**
  *Answer:* $R^2$ represents the proportion of variance in delivery time that is explained by the independent variables. An $R^2$ of 0.937 means 93.7% of variability in delivery times is captured by the model.